# SPHERE-3 Particle Classification Analysis

Analysis of LDF approximation parameters for cosmic ray particle identification (p / N / Fe).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet('results.parquet')
print(f'Loaded {len(df)} events')
print(f'Particles: {df["particle"].value_counts().to_dict()}')
print(f'Energies: {sorted(df["energy"].unique())}')
print(f'Angles: {sorted(df["angle"].unique())}')
print(f'Heights: {sorted(df["height"].unique())}')

In [ ]:
features = ['p0', 'p1', 'p4', 's', 'x0', 'y0', 'fval', 'rmse', 'r2',
            'Rc_snow', 'I_max', 'sum', 'Int', 'err_Int']

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for i, feat in enumerate(features):
    ax = axes[i // 4, i % 4]
    for particle in ['p', 'N', 'Fe']:
        subset = df[df['particle'] == particle][feat]
        ax.hist(subset, bins=50, alpha=0.5, label=particle, density=True)
    ax.set_title(feat)
    ax.legend()
for j in range(len(features), 16):
    axes[j // 4, j % 4].set_visible(False)
fig.tight_layout()
plt.show()

## Criterion Analysis

In [ ]:
from sphere_appro.criterion import run_criterion_parquet

run_criterion_parquet('results.parquet', 'analysis_output')
crit = pd.read_parquet('analysis_output/criterion_results.parquet')
print(f'Criterion results: {len(crit)} rows')
crit

## ML Classification

In [ ]:
from sphere_appro.ml_classification import load_and_split, train_and_evaluate

X_train, X_test, y_train, y_test, meta_test = load_and_split('results.parquet')
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

metrics = train_and_evaluate(X_train, X_test, y_train, y_test, meta_test, 'analysis_output')

In [ ]:
import json

with open('analysis_output/ml_metrics.json') as f:
    m = json.load(f)

print('Overall accuracy:')
for model, vals in m['overall'].items():
    print(f'  {model}: {vals["accuracy"]:.4f} (F1={vals["f1_macro"]:.4f})')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

results = pd.read_parquet('analysis_output/ml_results.parquet')
best_model = max(m['overall'], key=lambda k: m['overall'][k]['accuracy'])
y_true = results['y_true']
y_pred = results[f'y_pred_{best_model}']

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=['p', 'N', 'Fe'], ax=ax, cmap='Blues',
)
ax.set_title(f'Confusion Matrix ({best_model})')
plt.show()

In [ ]:
fi = m.get('feature_importances', {})
if fi:
    fi_df = pd.DataFrame(fi)
    fig, ax = plt.subplots(figsize=(12, 6))
    fi_df.plot(kind='bar', ax=ax)
    ax.set_title('Feature Importance Comparison')
    ax.set_ylabel('Importance')
    plt.xticks(rotation=45, ha='right')
    fig.tight_layout()
    plt.show()

## Separability Map

In [ ]:
from sphere_appro.separability_map import run_separability_analysis

table = run_separability_analysis('analysis_output')
table

## Conclusions

In [ ]:
print('=== Summary ===')
print(f'\nBest ML model: {best_model} (accuracy={m["overall"][best_model]["accuracy"]:.4f})')
print(f'\nCriterion (full dataset):')
full_crit = crit[crit['energy'] == 'all'].iloc[0]
print(f'  r1={full_crit["r1_opt"]:.0f}, r2={full_crit["r2_opt"]:.0f}')
print(f'  error p-N: {full_crit["error_pN"]:.4f}')
print(f'  error N-Fe: {full_crit["error_NFe"]:.4f}')
print(f'\nSeparability by slice (ML accuracy):')
for _, row in table.iterrows():
    best_acc = max(row.get('ml_accuracy_rf', 0) or 0,
                   row.get('ml_accuracy_xgb', 0) or 0,
                   row.get('ml_accuracy_lgbm', 0) or 0)
    print(f'  {row["energy"]}, {row["angle"]}\u00b0, {row["height"]}m: {best_acc:.4f}')